# Test de Hipótesis Central
## Efecto del Traslado Interhospitalario sobre la Mortalidad en Pacientes de Alta Complejidad
### Dataset: GRD Sector Público Chile 2024

---

**Proyecto:** Desigualdades en Derivaciones Hospitalarias (GRD)  
**Hipótesis central:**

> **H₀:** El traslado interhospitalario **no** tiene efecto estadísticamente significativo sobre la probabilidad de mortalidad en pacientes de alta complejidad (OR = 1; α = 0.05)  
> **H₁:** El traslado interhospitalario **aumenta** significativamente la probabilidad de mortalidad en pacientes de alta complejidad (OR > 1)

**Estructura:**
0. Setup e importación de datos  
1. Preparación de variables para el test  
2. Test 1 — Chi-cuadrado de independencia  
3. Test 2 — Test de Wald sobre coeficiente del traslado  
4. Test 3 — Likelihood Ratio Test (LRT)  
5. Test 4 — Mann-Whitney U (análisis complementario)  
6. Resumen e interpretación  

## 0. Setup e Importación de Datos

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path

# Tests estadísticos
from scipy import stats
from scipy.stats import chi2_contingency, mannwhitneyu, chi2

# Regresión logística con inferencia estadística
import statsmodels.api as sm
import statsmodels.formula.api as smf

# ─── Constantes del proyecto ───────────────────────────────────────────────
COLOR_PRINCIPAL = '#1B9EDB'   # azul corporativo ADIE
COLOR_ACENTO    = '#E05252'   # rojo para OR > 1 / rechazo H₀
COLOR_NEUTRO    = '#4CAF50'   # verde para OR ≤ 1
ALPHA           = 0.05        # nivel de significancia
RANDOM_STATE    = 42

DATA_PATH   = '../data/archivosDuros/GRD_PUBLICO_2024.txt'
PLOTS_DIR   = Path('../plots')
DATA_DIR    = Path('../data/processed')
for d in [PLOTS_DIR, DATA_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print('Setup completo.')
print(f'Nivel de significancia α = {ALPHA}')

In [ ]:
# Carga con múltiples encodings por robustez
df_raw = None
encodings_a_probar = ['UTF-16', 'UTF-8-sig', 'latin-1']

for enc in encodings_a_probar:
    try:
        df_raw = pd.read_csv(
            DATA_PATH,
            sep='|',
            encoding=enc,
            dtype=str,
            low_memory=False
        )
        print(f'✓ Archivo leído correctamente con encoding: {enc}')
        break
    except Exception as e:
        print(f'  Encoding {enc} falló: {type(e).__name__}')

if df_raw is None:
    raise RuntimeError('No se pudo cargar el archivo con ningún encoding.')

print(f'Shape original: {df_raw.shape}')

# Eliminar fila duplicada de headers si existe
if (df_raw.iloc[0] == df_raw.columns).mean() > 0.5:
    print('Fila de headers duplicados detectada — eliminando...')
    df_raw = df_raw.iloc[1:].reset_index(drop=True)

df = df_raw.copy()
print(f'Shape final: {df.shape}')
print('\nColumnas GRD relevantes:')
cols_grd = [c for c in df.columns if any(k in c.upper() for k in
            ['SEVERIDAD','MORTALIDAD','HOSPPROCEDENCIA','PESO','TIPO_PROCEDENCIA'])]
for c in cols_grd:
    print(f'  {c}')

## 1. Preparación de Variables para el Test

Se construyen las variables analíticas a partir de los campos crudos del GRD:

| Variable | Derivación | Descripción |
|---|---|---|
| `FUE_TRASLADADO` | `HOSPPROCEDENCIA` no nulo/vacío | 1 si el paciente provino de otro recinto hospitalario |
| `ALTA_SEVERIDAD` | `IR_29301_SEVERIDAD` ≥ 3 | 1 si el paciente es de alta complejidad clínica |
| `MORTALIDAD_BINARIA` | `IR_29301_MORTALIDAD` > 2 | 1 si el riesgo de mortalidad GRD es máximo (categoría 3) |
| `PESO_GRD` | `IR_29301_PESO` numérico | Consumo relativo de recursos del episodio |
| `REGION_COD` | `SERVICIO_SALUD` → región | Código de región para control de heterogeneidad geográfica |

In [ ]:
# ── 1.1 FUE_TRASLADADO ────────────────────────────────────────────────────
# La columna HOSPPROCEDENCIA contiene el nombre del hospital emisor del traslado.
# Si no es nula ni vacía, el paciente provino de otro recinto.
hosp_proc = df['HOSPPROCEDENCIA'].astype(str).str.strip()
df['FUE_TRASLADADO'] = (~hosp_proc.isin(['', 'nan', 'NaN', 'NAN'])).astype(int)

n_traslados = df['FUE_TRASLADADO'].sum()
print(f'FUE_TRASLADADO = 1: {n_traslados:,} ({100*n_traslados/len(df):.2f}%)')
print(f'FUE_TRASLADADO = 0: {len(df)-n_traslados:,} ({100*(1-n_traslados/len(df)):.2f}%)')

# ── 1.2 IR_29301_SEVERIDAD numérica ───────────────────────────────────────
df['SEVERIDAD'] = pd.to_numeric(
    df['IR_29301_SEVERIDAD'].str.strip(), errors='coerce'
).fillna(0).astype(int).clip(0, 3)

print('\nDistribución IR_29301_SEVERIDAD:')
print(df['SEVERIDAD'].value_counts().sort_index().to_dict())

# ── 1.3 ALTA_SEVERIDAD ────────────────────────────────────────────────────
df['ALTA_SEVERIDAD'] = (df['SEVERIDAD'] >= 3).astype(int)
n_alta = df['ALTA_SEVERIDAD'].sum()
print(f'\nALTA_SEVERIDAD = 1: {n_alta:,} ({100*n_alta/len(df):.2f}%)')

# ── 1.4 IR_29301_MORTALIDAD numérica ──────────────────────────────────────
df['MORTALIDAD_SCORE'] = pd.to_numeric(
    df['IR_29301_MORTALIDAD'].str.strip(), errors='coerce'
).fillna(0).astype(int).clip(0, 3)

print('\nDistribución IR_29301_MORTALIDAD:')
print(df['MORTALIDAD_SCORE'].value_counts().sort_index().to_dict())

# ── 1.5 MORTALIDAD_BINARIA ────────────────────────────────────────────────
# Umbral > 2 → categoría 3 (máximo riesgo de mortalidad según el sistema GRD)
df['MORTALIDAD_BINARIA'] = (df['MORTALIDAD_SCORE'] > 2).astype(int)
n_mort = df['MORTALIDAD_BINARIA'].sum()
print(f'\nMORTALIDAD_BINARIA = 1: {n_mort:,} ({100*n_mort/len(df):.2f}%)')

# ── 1.6 PESO_GRD numérico ─────────────────────────────────────────────────
df['PESO_GRD'] = pd.to_numeric(
    df['IR_29301_PESO'].str.strip().str.replace(',', '.', regex=False),
    errors='coerce'
).fillna(0.0)

print(f'\nPESO_GRD — media: {df["PESO_GRD"].mean():.3f}, mediana: {df["PESO_GRD"].median():.3f}')

In [ ]:
# ── 1.7 REGION_COD desde SERVICIO_SALUD ───────────────────────────────────
REGION_MAP = {
    'ARICA'                    : 'R15_Arica',
    'IQUIQUE'                  : 'R01_Tarapaca',
    'ANTOFAGASTA'              : 'R02_Antofagasta',
    'ATACAMA'                  : 'R03_Atacama',
    'COQUIMBO'                 : 'R04_Coquimbo',
    'VALPARAISO SAN ANTONIO'   : 'R05_Valparaiso',
    'VIÑA DEL MAR QUILLOTA'    : 'R05_Valparaiso',
    'ACONCAGUA'                : 'R05_Valparaiso',
    'METROPOLITANO CENTRAL'    : 'R13_Metropolitana',
    'METROPOLITANO NORTE'      : 'R13_Metropolitana',
    'METROPOLITANO ORIENTE'    : 'R13_Metropolitana',
    'METROPOLITANO OCCIDENTE'  : 'R13_Metropolitana',
    'METROPOLITANO SUR'        : 'R13_Metropolitana',
    'METROPOLITANO SURORIENTE' : 'R13_Metropolitana',
    'LIBERTADOR B. O HIGGINS'  : 'R06_OHiggins',
    'DEL MAULE'                : 'R07_Maule',
    'ÑUBLE'                    : 'R16_Nuble',
    'CONCEPCIÓN'               : 'R08_Biobio',
    'TALCAHUANO'               : 'R08_Biobio',
    'BIOBIO'                   : 'R08_Biobio',
    'ARAUCO'                   : 'R08_Biobio',
    'ARAUCANÍA SUR'            : 'R09_Araucania',
    'ARAUCANÍA NORTE'          : 'R09_Araucania',
    'VALDIVIA'                 : 'R14_LosRios',
    'DEL RELONCAVÍ'            : 'R10_LosLagos',
    'OSORNO'                   : 'R10_LosLagos',
    'CHILOÉ'                   : 'R10_LosLagos',
    'AYSEN'                    : 'R11_Aysen',
    'MAGALLANES'               : 'R12_Magallanes',
}

df['_SS'] = df['SERVICIO_SALUD'].str.strip().str.upper()

# Mapeo insensible a mayúsculas
region_map_upper = {k.upper(): v for k, v in REGION_MAP.items()}
df['REGION_COD'] = df['_SS'].map(region_map_upper).fillna('R99_Otra')

print('Distribución REGION_COD:')
print(df['REGION_COD'].value_counts().to_string())

In [ ]:
# ── 1.8 Estadísticas descriptivas del subconjunto ALTA_SEVERIDAD = 1 ──────
df_alta = df[df['ALTA_SEVERIDAD'] == 1].copy()
print(f'Pacientes ALTA_SEVERIDAD=1: {len(df_alta):,}')
print(f'  - FUE_TRASLADADO=1: {df_alta["FUE_TRASLADADO"].sum():,} '
      f'({100*df_alta["FUE_TRASLADADO"].mean():.2f}%)')
print(f'  - MORTALIDAD_BINARIA=1: {df_alta["MORTALIDAD_BINARIA"].sum():,} '
      f'({100*df_alta["MORTALIDAD_BINARIA"].mean():.2f}%)')
print()

# Tasas de mortalidad por grupo de traslado (dentro de ALTA_SEVERIDAD)
tab_desc = df_alta.groupby('FUE_TRASLADADO')['MORTALIDAD_BINARIA'].agg(
    N='count',
    N_mortalidad_alta='sum',
    Tasa_mortalidad='mean'
).rename(index={0: 'No trasladado', 1: 'Fue trasladado'})
tab_desc['Tasa_%'] = (tab_desc['Tasa_mortalidad'] * 100).round(2)
print('Mortalidad alta por grupo (ALTA_SEVERIDAD=1):')
print(tab_desc.to_string())

## 2. Test 1: Chi-cuadrado de Independencia

### Planteamiento

Antes de cualquier modelo, se evalúa si existe asociación estadística entre el traslado y la mortalidad alta **usando solo pacientes de alta complejidad (ALTA_SEVERIDAD = 1)**, sin ajuste por covariables.

- **Variables:** `FUE_TRASLADADO` × `MORTALIDAD_BINARIA`  
- **Muestra:** Pacientes con `ALTA_SEVERIDAD = 1`  
- **Estadístico:** χ² de Pearson con corrección de Yates si n < 5 en alguna celda  
- **H₀:** Las variables son estadísticamente independientes  
- **Decisión:** Rechazar H₀ si p-value < α = 0.05

In [ ]:
# ── Tabla de contingencia ─────────────────────────────────────────────────
tabla_contingencia = pd.crosstab(
    df_alta['FUE_TRASLADADO'],
    df_alta['MORTALIDAD_BINARIA'],
    rownames=['FUE_TRASLADADO'],
    colnames=['MORTALIDAD_BINARIA']
)
tabla_contingencia.index = ['No trasladado (0)', 'Fue trasladado (1)']
tabla_contingencia.columns = ['Mortalidad baja (0)', 'Mortalidad alta (1)']

print('Tabla de contingencia (frecuencias absolutas):')
print(tabla_contingencia.to_string())
print()

# Proporciones por fila
tabla_props = tabla_contingencia.div(tabla_contingencia.sum(axis=1), axis=0) * 100
print('Tabla de proporciones (% dentro de cada fila):')
print(tabla_props.round(2).to_string())

In [ ]:
# ── Test χ² ───────────────────────────────────────────────────────────────
obs = tabla_contingencia.values  # matriz 2x2 de frecuencias observadas
chi2_stat, p_chi2, dof, expected = chi2_contingency(obs, correction=False)

# Verificar supuesto: frecuencia esperada ≥ 5 en todas las celdas
min_expected = expected.min()
usar_yates = min_expected < 5

if usar_yates:
    chi2_stat, p_chi2, dof, expected = chi2_contingency(obs, correction=True)
    print('Nota: Se aplicó corrección de Yates (frecuencia esperada < 5)')
else:
    print('Supuesto χ² verificado: todas las frecuencias esperadas ≥ 5')

decision_chi2 = 'RECHAZAR H₀' if p_chi2 < ALPHA else 'NO RECHAZAR H₀'

print(f'\n{"="*55}')
print('TEST 1 — Chi-cuadrado de independencia')
print(f'{"="*55}')
print(f'  χ²({dof})       = {chi2_stat:.4f}')
print(f'  p-value         = {p_chi2:.6f}')
print(f'  α               = {ALPHA}')
print(f'  Decisión        : {decision_chi2}')
print(f'  Min. frec. esp. = {min_expected:.2f}')

# Phi coefficient (tamaño de efecto para tabla 2x2)
n_total_chi2 = obs.sum()
phi = np.sqrt(chi2_stat / n_total_chi2)
print(f'  Phi (efecto)    = {phi:.4f}  (|φ| < 0.1 pequeño, 0.1–0.3 mediano, > 0.3 grande)')

In [ ]:
# ── Visualización: barras agrupadas con proporciones ──────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Panel izquierdo: proporciones de mortalidad alta por grupo
grupos = ['No trasladado', 'Fue trasladado']
pct_mort = [
    100 * obs[0, 1] / obs[0].sum(),
    100 * obs[1, 1] / obs[1].sum()
]
pct_no_mort = [100 - p for p in pct_mort]

x = np.arange(len(grupos))
width = 0.4
bars1 = axes[0].bar(x - width/2, pct_no_mort, width, label='Mortalidad baja',
                    color=COLOR_PRINCIPAL, alpha=0.85, edgecolor='white')
bars2 = axes[0].bar(x + width/2, pct_mort, width, label='Mortalidad alta',
                    color=COLOR_ACENTO, alpha=0.85, edgecolor='white')

for bar, val in zip(bars1, pct_no_mort):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                 f'{val:.1f}%', ha='center', va='bottom', fontsize=9)
for bar, val in zip(bars2, pct_mort):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                 f'{val:.1f}%', ha='center', va='bottom', fontsize=9)

axes[0].set_xticks(x)
axes[0].set_xticklabels(grupos, fontsize=10)
axes[0].set_ylabel('Proporción (%)')
axes[0].set_title('Distribución de Mortalidad por Grupo de Traslado\n(ALTA_SEVERIDAD = 1)', fontsize=10)
axes[0].legend(fontsize=9)
axes[0].set_ylim(0, max(pct_no_mort) * 1.12)
axes[0].grid(axis='y', alpha=0.3)

# Panel derecho: mapa de calor de la tabla de contingencia normalizada
tabla_heat = tabla_contingencia.copy().astype(float)
for col in tabla_heat.columns:
    tabla_heat[col] = tabla_heat[col] / tabla_heat[col].sum()

sns.heatmap(
    tabla_props / 100,
    annot=True, fmt='.3f', cmap='Blues',
    ax=axes[1], linewidths=0.5,
    cbar_kws={'label': 'Proporción dentro del grupo'}
)
axes[1].set_title(
    f'Tabla de Contingencia (proporciones por fila)\n'
    f'χ²={chi2_stat:.2f}, p={p_chi2:.4f} → {decision_chi2}',
    fontsize=10
)
axes[1].set_xlabel('MORTALIDAD_BINARIA')
axes[1].set_ylabel('FUE_TRASLADADO')

plt.tight_layout()
plt.savefig(PLOTS_DIR / 'test1_chi2_traslado_mortalidad.png', dpi=150, bbox_inches='tight')
plt.show()
print('Guardado: plots/test1_chi2_traslado_mortalidad.png')

## 3. Test 2: Test de Wald sobre el Coeficiente de FUE_TRASLADADO

### Planteamiento

Se ajusta un modelo de regresión logística con `statsmodels` para obtener inferencia estadística formal. El **test de Wald** evalúa si el coeficiente β asociado a `FUE_TRASLADADO` es significativamente diferente de cero.

**Modelo ajustado (pacientes ALTA_SEVERIDAD = 1):**
$$\log\!\left(\frac{P(\text{Mortalidad alta})}{1 - P(\text{Mortalidad alta})}\right) = \beta_0 + \beta_1 \cdot \text{FUE\_TRASLADADO} + \beta_2 \cdot \text{PESO\_GRD} + \sum_r \beta_r \cdot \mathbf{1}[\text{Región} = r]$$

**Estadístico de Wald:**
$$z = \frac{\hat{\beta}}{\widehat{\text{SE}}(\hat{\beta})} \sim \mathcal{N}(0, 1) \text{ bajo } H_0$$

**Odds Ratio:** $\text{OR} = e^{\hat{\beta}}$  con IC 95%: $\left[e^{\hat{\beta} - 1.96 \cdot \text{SE}},\ e^{\hat{\beta} + 1.96 \cdot \text{SE}}\right]$

In [ ]:
# ── Preparar dataset para statsmodels ─────────────────────────────────────
# Usar pacientes de alta severidad; REGION_COD como variable categórica
df_model = df_alta[['MORTALIDAD_BINARIA', 'FUE_TRASLADADO', 'PESO_GRD', 'REGION_COD']].dropna().copy()

# Región de referencia: la más frecuente (Metropolitana)
ref_region = df_model['REGION_COD'].value_counts().idxmax()
df_model['REGION_COD'] = pd.Categorical(
    df_model['REGION_COD'],
    categories=[ref_region] + [r for r in df_model['REGION_COD'].unique() if r != ref_region]
)

print(f'Dataset para modelos: {df_model.shape}')
print(f'Región de referencia: {ref_region}')
print(f'Tasa MORTALIDAD_BINARIA=1: {df_model["MORTALIDAD_BINARIA"].mean()*100:.2f}%')
print(f'Tasa FUE_TRASLADADO=1: {df_model["FUE_TRASLADADO"].mean()*100:.2f}%')

# Fórmula del modelo completo
formula_completo = 'MORTALIDAD_BINARIA ~ FUE_TRASLADADO + PESO_GRD + C(REGION_COD)'

print('\nAjustando modelo completo (statsmodels Logit)...')
modelo_completo = smf.logit(formula_completo, data=df_model).fit(
    method='bfgs',
    maxiter=200,
    disp=False
)
print('Modelo completo ajustado.')
print(f'  Log-likelihood: {modelo_completo.llf:.4f}')
print(f'  AIC: {modelo_completo.aic:.4f}')
print(f'  Converged: {modelo_completo.mle_retvals["converged"] if hasattr(modelo_completo.mle_retvals, "__getitem__") else "N/A"}')

In [ ]:
# ── Extraer coeficiente β de FUE_TRASLADADO y realizar test de Wald ───────
beta_traslado = modelo_completo.params['FUE_TRASLADADO']
se_traslado   = modelo_completo.bse['FUE_TRASLADADO']
z_score       = modelo_completo.tvalues['FUE_TRASLADADO']   # z = β / SE
p_wald        = modelo_completo.pvalues['FUE_TRASLADADO']   # p-value bilateral

# IC 95% en escala log-odds
ci = modelo_completo.conf_int()
ci_lower_beta = ci.loc['FUE_TRASLADADO', 0]
ci_upper_beta = ci.loc['FUE_TRASLADADO', 1]

# Odds Ratio y su IC 95%
OR          = np.exp(beta_traslado)
OR_ci_lower = np.exp(ci_lower_beta)
OR_ci_upper = np.exp(ci_upper_beta)

decision_wald = 'RECHAZAR H₀' if p_wald < ALPHA else 'NO RECHAZAR H₀'

print(f'{"="*55}')
print('TEST 2 — Test de Wald sobre β(FUE_TRASLADADO)')
print(f'{"="*55}')
print(f'  Coeficiente β   = {beta_traslado:.6f}')
print(f'  Error estándar  = {se_traslado:.6f}')
print(f'  z-score (Wald)  = {z_score:.4f}')
print(f'  p-value (bilat) = {p_wald:.6f}')
print(f'  IC 95% [β_L, β_U] = [{ci_lower_beta:.4f}, {ci_upper_beta:.4f}]')
print()
print(f'  OR = exp(β)     = {OR:.4f}')
print(f'  OR IC 95%       = [{OR_ci_lower:.4f}, {OR_ci_upper:.4f}]')
print(f'  α               = {ALPHA}')
print(f'  Decisión        : {decision_wald}')

# Resumen de todos los coeficientes del modelo
print('\nResumen completo del modelo:')
print(modelo_completo.summary2().tables[1].to_string())

In [ ]:
# ── Forest Plot del OR con IC 95% ─────────────────────────────────────────
# Incluir todos los coeficientes del modelo (excepto el intercepto y las regiones)
params = modelo_completo.params
ci_all = modelo_completo.conf_int()
pvals  = modelo_completo.pvalues

# Seleccionar coeficientes relevantes (no intercepto, mostrar top regiones)
vars_interes = ['FUE_TRASLADADO', 'PESO_GRD']
vars_region  = [v for v in params.index if 'REGION_COD' in v][:5]  # top 5 regiones
vars_plot    = vars_interes + vars_region

# Limpiar etiquetas para visualización
def limpiar_etiqueta(s):
    return (s.replace('C(REGION_COD)[T.', '').replace(']', '')
             .replace('_', ' '))

etiquetas  = [limpiar_etiqueta(v) for v in vars_plot]
ors        = np.exp(params[vars_plot].values)
ci_lowers  = np.exp(ci_all.loc[vars_plot, 0].values)
ci_uppers  = np.exp(ci_all.loc[vars_plot, 1].values)
significat = pvals[vars_plot].values < ALPHA

fig, ax = plt.subplots(figsize=(9, max(4, len(vars_plot) * 0.65 + 1)))

y_pos = np.arange(len(vars_plot))
colors_plot = [
    COLOR_ACENTO if (o > 1 and s) else COLOR_PRINCIPAL if s else '#AAAAAA'
    for o, s in zip(ors, significat)
]

ax.scatter(ors, y_pos, color=colors_plot, zorder=3, s=80)
ax.errorbar(
    ors, y_pos,
    xerr=[ors - ci_lowers, ci_uppers - ors],
    fmt='none', color='#555555', capsize=5, lw=1.5, zorder=2
)

# Destacar FUE_TRASLADADO
idx_ft = vars_plot.index('FUE_TRASLADADO')
ax.annotate(
    f'OR = {OR:.3f}\nIC 95%: [{OR_ci_lower:.3f}, {OR_ci_upper:.3f}]\np = {p_wald:.4f}',
    xy=(OR, idx_ft),
    xytext=(max(ors) * 0.75, idx_ft + 0.8),
    fontsize=8.5,
    bbox=dict(boxstyle='round,pad=0.3', facecolor='#FFF9C4', alpha=0.8),
    arrowprops=dict(arrowstyle='->', color='#444', lw=1)
)

ax.axvline(1.0, color='black', linestyle='--', lw=1.0, label='OR = 1 (H₀)')
ax.set_yticks(y_pos)
ax.set_yticklabels(etiquetas, fontsize=9)
ax.set_xlabel('Odds Ratio (IC 95%)', fontsize=10)
ax.set_title(
    'Forest Plot — OR del Modelo Logístico\n(MORTALIDAD_BINARIA ~ FUE_TRASLADADO + PESO_GRD + Región)',
    fontsize=10
)

# Leyenda de colores
legend_elements = [
    mpatches.Patch(facecolor=COLOR_ACENTO, label='OR > 1, p < α'),
    mpatches.Patch(facecolor=COLOR_PRINCIPAL, label='OR ≤ 1, p < α'),
    mpatches.Patch(facecolor='#AAAAAA', label='No significativo'),
]
ax.legend(handles=legend_elements, fontsize=8, loc='lower right')
ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig(PLOTS_DIR / 'test2_forest_plot_OR.png', dpi=150, bbox_inches='tight')
plt.show()
print('Guardado: plots/test2_forest_plot_OR.png')

## 4. Test 3: Likelihood Ratio Test (LRT)

### Planteamiento

El **Likelihood Ratio Test** compara la verosimilitud de un modelo completo versus uno restringido (sin `FUE_TRASLADADO`). Este test es más robusto que el test de Wald en presencia de separación cuasi-completa o coeficientes grandes.

$$\text{LRT} = -2 \left[ \ell(\text{Modelo restringido}) - \ell(\text{Modelo completo}) \right] \sim \chi^2(\text{gl}=1) \text{ bajo } H_0$$

- **Modelo completo:** `MORTALIDAD_BINARIA ~ FUE_TRASLADADO + PESO_GRD + C(REGION_COD)`  
- **Modelo restringido:** `MORTALIDAD_BINARIA ~ PESO_GRD + C(REGION_COD)` *(sin FUE_TRASLADADO)*  
- **gl = 1** (un parámetro adicional en el modelo completo)

Un LRT significativo indica que incluir `FUE_TRASLADADO` mejora significativamente el ajuste del modelo.

In [ ]:
# ── Ajustar modelo RESTRINGIDO (sin FUE_TRASLADADO) ───────────────────────
formula_restringido = 'MORTALIDAD_BINARIA ~ PESO_GRD + C(REGION_COD)'

print('Ajustando modelo restringido...')
modelo_restringido = smf.logit(formula_restringido, data=df_model).fit(
    method='bfgs',
    maxiter=200,
    disp=False
)
print(f'  Log-likelihood (restringido): {modelo_restringido.llf:.4f}')
print(f'  Log-likelihood (completo)  : {modelo_completo.llf:.4f}')

# ── Estadístico LRT ───────────────────────────────────────────────────────
# LRT = -2 * (llf_restringido - llf_completo)
# El valor debe ser positivo porque llf_completo ≥ llf_restringido
lrt_stat = -2 * (modelo_restringido.llf - modelo_completo.llf)
lrt_df   = modelo_completo.df_model - modelo_restringido.df_model  # grados de libertad
p_lrt    = chi2.sf(lrt_stat, df=lrt_df)  # p-value de la cola derecha

decision_lrt = 'RECHAZAR H₀' if p_lrt < ALPHA else 'NO RECHAZAR H₀'

print(f'\n{"="*55}')
print('TEST 3 — Likelihood Ratio Test (LRT)')
print(f'{"="*55}')
print(f'  LLF (completo)    = {modelo_completo.llf:.4f}')
print(f'  LLF (restringido) = {modelo_restringido.llf:.4f}')
print(f'  Diferencia LLF    = {modelo_completo.llf - modelo_restringido.llf:.4f}')
print(f'  LRT = -2·ΔLLF    = {lrt_stat:.4f}')
print(f'  Grados de libertad= {lrt_df:.0f}')
print(f'  p-value           = {p_lrt:.6f}')
print(f'  α                 = {ALPHA}')
print(f'  Decisión          : {decision_lrt}')

# Comparar AIC y BIC
print(f'\n  AIC (completo)    = {modelo_completo.aic:.2f}')
print(f'  AIC (restringido) = {modelo_restringido.aic:.2f}')
print(f'  ΔAIC              = {modelo_restringido.aic - modelo_completo.aic:.2f}')

In [ ]:
# ── Visualización: distribución χ² y posición del estadístico LRT ─────────
fig, ax = plt.subplots(figsize=(8, 4.5))

# Distribución χ²(1)
x_chi2 = np.linspace(0, max(lrt_stat * 1.5, 10), 300)
y_chi2 = chi2.pdf(x_chi2, df=1)
ax.plot(x_chi2, y_chi2, color='#333333', lw=2, label='χ²(1) bajo H₀')

# Zona de rechazo (cola derecha)
valor_critico = chi2.ppf(1 - ALPHA, df=1)
x_rechazo = np.linspace(valor_critico, x_chi2.max(), 100)
ax.fill_between(x_rechazo, chi2.pdf(x_rechazo, df=1),
                color=COLOR_ACENTO, alpha=0.3, label=f'Zona rechazo (α={ALPHA})')

# Estadístico LRT observado
ax.axvline(lrt_stat, color=COLOR_ACENTO, lw=2.5, linestyle='--',
           label=f'LRT = {lrt_stat:.2f}')
ax.axvline(valor_critico, color=COLOR_PRINCIPAL, lw=1.5, linestyle=':',
           label=f'Valor crítico = {valor_critico:.2f}')

ax.text(lrt_stat, ax.get_ylim()[1] * 0.85,
        f' p = {p_lrt:.5f}\n → {decision_lrt}',
        color=COLOR_ACENTO, fontsize=9, va='top')

ax.set_xlabel('Estadístico LRT', fontsize=10)
ax.set_ylabel('Densidad de probabilidad', fontsize=10)
ax.set_title('Test 3 — Distribución χ²(1) y estadístico LRT\n(Completo vs. Restringido sin FUE_TRASLADADO)',
             fontsize=10)
ax.legend(fontsize=9)
ax.set_xlim(left=0)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(PLOTS_DIR / 'test3_lrt_chi2.png', dpi=150, bbox_inches='tight')
plt.show()
print('Guardado: plots/test3_lrt_chi2.png')

## 5. Test 4: Mann-Whitney U — Análisis Complementario

### Planteamiento

El **test de Mann-Whitney U** (Wilcoxon de dos muestras) compara la distribución del `PESO_GRD` entre pacientes trasladados y no trasladados, sin asumir normalidad. `PESO_GRD` es un proxy del consumo relativo de recursos y de la complejidad real del caso.

> **Hipótesis complementaria:** Si los pacientes trasladados tienen mayor PESO_GRD que los no trasladados, esto sugiere que los traslados interhospitalarios corresponden a casos genuinamente más complejos, lo que refuerza la interpretación del mayor riesgo de mortalidad.

- **H₀:** Las distribuciones de PESO_GRD son iguales entre grupos (U-statistic ~ distribución nula)  
- **H₁:** Los trasladados tienen mayor PESO_GRD (prueba unilateral)

*Nota: Se usa el dataset completo (no solo ALTA_SEVERIDAD) para mayor potencia estadística.*

In [ ]:
# ── Datos para Mann-Whitney (dataset completo) ────────────────────────────
mask_valido = (df['PESO_GRD'] > 0) & df['FUE_TRASLADADO'].notna()
df_mw = df[mask_valido].copy()

grupo_trasl    = df_mw.loc[df_mw['FUE_TRASLADADO'] == 1, 'PESO_GRD']
grupo_no_trasl = df_mw.loc[df_mw['FUE_TRASLADADO'] == 0, 'PESO_GRD']

mediana_trasl    = grupo_trasl.median()
mediana_no_trasl = grupo_no_trasl.median()
media_trasl      = grupo_trasl.mean()
media_no_trasl   = grupo_no_trasl.mean()

print(f'Grupo TRASLADADO     : n={len(grupo_trasl):,} | mediana={mediana_trasl:.4f} | media={media_trasl:.4f}')
print(f'Grupo NO TRASLADADO  : n={len(grupo_no_trasl):,} | mediana={mediana_no_trasl:.4f} | media={media_no_trasl:.4f}')

# ── Test Mann-Whitney U (alternativa: 'greater' para H1: trasladados > no trasladados)
u_stat, p_mw = mannwhitneyu(
    grupo_trasl,
    grupo_no_trasl,
    alternative='greater'  # H1: distribución de trasladados estocasticamente mayor
)

# Tamaño de efecto: rango biserial de correlación
n1, n2 = len(grupo_trasl), len(grupo_no_trasl)
r_biserial = 1 - (2 * u_stat) / (n1 * n2)   # varía entre -1 y 1

decision_mw = 'RECHAZAR H₀' if p_mw < ALPHA else 'NO RECHAZAR H₀'

print(f'\n{"="*55}')
print('TEST 4 — Mann-Whitney U')
print(f'{"="*55}')
print(f'  U statistic    = {u_stat:.4f}')
print(f'  p-value (unilat, greater) = {p_mw:.6f}')
print(f'  r biserial (efecto) = {r_biserial:.4f}  (|r| < 0.1 trivial, 0.1–0.3 pequeño, > 0.3 mediano)')
print(f'  α              = {ALPHA}')
print(f'  Decisión       : {decision_mw}')
print()
print(f'  Mediana PESO_GRD (trasladado)    = {mediana_trasl:.4f}')
print(f'  Mediana PESO_GRD (no trasladado) = {mediana_no_trasl:.4f}')
print(f'  Diferencia medianas              = {mediana_trasl - mediana_no_trasl:+.4f}')

In [ ]:
# ── Visualización: boxplot comparativo y distribución de densidad ─────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Panel izquierdo: boxplot
PESO_CAP = df_mw['PESO_GRD'].quantile(0.95)  # capear outliers extremos para visualización

grupos_plot = [
    grupo_no_trasl.clip(upper=PESO_CAP),
    grupo_trasl.clip(upper=PESO_CAP)
]
bp = axes[0].boxplot(
    grupos_plot,
    labels=['No trasladado', 'Fue trasladado'],
    patch_artist=True,
    medianprops=dict(color='#111111', linewidth=2),
    whiskerprops=dict(lw=1.5),
    capprops=dict(lw=1.5),
    flierprops=dict(marker='.', markersize=2, alpha=0.3)
)
bp['boxes'][0].set_facecolor(COLOR_PRINCIPAL)
bp['boxes'][0].set_alpha(0.75)
bp['boxes'][1].set_facecolor(COLOR_ACENTO)
bp['boxes'][1].set_alpha(0.75)

axes[0].set_ylabel('PESO_GRD (capeado en P95)')
axes[0].set_title(
    f'Distribución PESO_GRD por Grupo\n'
    f'U = {u_stat:.0f}, p = {p_mw:.4f} → {decision_mw}',
    fontsize=10
)
axes[0].grid(axis='y', alpha=0.3)

# Anotar medianas
for i, (mediana, lab) in enumerate([
    (mediana_no_trasl, 'No trasladado'),
    (mediana_trasl, 'Fue trasladado')
]):
    axes[0].text(i + 1, mediana + 0.05, f'Mediana:\n{mediana:.3f}',
                 ha='center', va='bottom', fontsize=8.5,
                 bbox=dict(boxstyle='round,pad=0.2', fc='white', alpha=0.7))

# Panel derecho: distribuciones de densidad (KDE)
peso_cap_plot = PESO_CAP
datos_kde_0 = grupo_no_trasl[grupo_no_trasl <= peso_cap_plot]
datos_kde_1 = grupo_trasl[grupo_trasl <= peso_cap_plot]

axes[1].hist(datos_kde_0, bins=60, density=True, alpha=0.5,
             color=COLOR_PRINCIPAL, label=f'No trasladado (n={len(grupo_no_trasl):,})')
axes[1].hist(datos_kde_1, bins=60, density=True, alpha=0.5,
             color=COLOR_ACENTO, label=f'Fue trasladado (n={len(grupo_trasl):,})')
axes[1].axvline(mediana_no_trasl, color=COLOR_PRINCIPAL, lw=2, linestyle='--',
                label=f'Mediana NT = {mediana_no_trasl:.3f}')
axes[1].axvline(mediana_trasl, color=COLOR_ACENTO, lw=2, linestyle='--',
                label=f'Mediana FT = {mediana_trasl:.3f}')
axes[1].set_xlabel('PESO_GRD')
axes[1].set_ylabel('Densidad')
axes[1].set_title('Distribución de PESO_GRD por Grupo\n(Histograma de densidad, capeado en P95)', fontsize=10)
axes[1].legend(fontsize=8.5)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(PLOTS_DIR / 'test4_mannwhitney_peso_grd.png', dpi=150, bbox_inches='tight')
plt.show()
print('Guardado: plots/test4_mannwhitney_peso_grd.png')

## 6. Resumen e Interpretación

### Tabla Resumen de los 4 Tests

In [ ]:
# ── Construir tabla resumen ────────────────────────────────────────────────
resumen_tests = pd.DataFrame([
    {
        'Test': 'Test 1 — Chi-cuadrado',
        'Descripción': 'Independencia FUE_TRASLADADO × MORTALIDAD_BINARIA (ALTA_SEVERIDAD=1)',
        'Estadístico': f'χ²({dof}) = {chi2_stat:.4f}',
        'Valor_estadístico': round(chi2_stat, 4),
        'p_value': round(p_chi2, 6),
        'Decisión': decision_chi2,
        'Efecto': f'φ = {phi:.4f}'
    },
    {
        'Test': 'Test 2 — Wald',
        'Descripción': 'Significancia del coeficiente β(FUE_TRASLADADO) en modelo logístico',
        'Estadístico': f'z = {z_score:.4f}',
        'Valor_estadístico': round(z_score, 4),
        'p_value': round(p_wald, 6),
        'Decisión': decision_wald,
        'Efecto': f'OR = {OR:.4f} IC95%[{OR_ci_lower:.3f}, {OR_ci_upper:.3f}]'
    },
    {
        'Test': 'Test 3 — LRT',
        'Descripción': 'Mejora del modelo al incluir FUE_TRASLADADO (completo vs. restringido)',
        'Estadístico': f'LRT = {lrt_stat:.4f} (gl={lrt_df:.0f})',
        'Valor_estadístico': round(lrt_stat, 4),
        'p_value': round(p_lrt, 6),
        'Decisión': decision_lrt,
        'Efecto': f'ΔAIC = {modelo_restringido.aic - modelo_completo.aic:.2f}'
    },
    {
        'Test': 'Test 4 — Mann-Whitney U',
        'Descripción': 'Distribución PESO_GRD: trasladados vs. no trasladados',
        'Estadístico': f'U = {u_stat:.4f}',
        'Valor_estadístico': round(u_stat, 4),
        'p_value': round(p_mw, 6),
        'Decisión': decision_mw,
        'Efecto': f'r biserial = {r_biserial:.4f}; ΔMediana = {mediana_trasl - mediana_no_trasl:+.4f}'
    },
])

print('TABLA RESUMEN — 4 TESTS DE HIPÓTESIS')
print('='*80)
for _, row in resumen_tests.iterrows():
    simbolo = '✓ ' if 'RECHAZAR' in row['Decisión'] else '✗ '
    print(f"\n{simbolo}{row['Test']}")
    print(f"   {row['Descripción']}")
    print(f"   Estadístico : {row['Estadístico']}")
    print(f"   p-value     : {row['p_value']:.6f}")
    print(f"   Decisión    : {row['Decisión']}")
    print(f"   Efecto      : {row['Efecto']}")
print('\n' + '='*80)

In [ ]:
# ── Visualización: panel resumen de decisiones ────────────────────────────
fig, ax = plt.subplots(figsize=(10, 4.5))

nombres_tests = [
    f'Test 1\nChi-cuadrado\nχ²={chi2_stat:.2f}',
    f'Test 2\nWald\nz={z_score:.2f}',
    f'Test 3\nLRT\nLRT={lrt_stat:.2f}',
    f'Test 4\nMann-Whitney\nU={u_stat:.0f}'
]
pvalores = [p_chi2, p_wald, p_lrt, p_mw]
decisiones = [decision_chi2, decision_wald, decision_lrt, decision_mw]

colores_dec = [COLOR_ACENTO if 'RECHAZAR' in d else '#AAAAAA' for d in decisiones]

bars = ax.bar(
    range(4), [-np.log10(max(p, 1e-300)) for p in pvalores],
    color=colores_dec, edgecolor='white', width=0.6
)

# Línea de significancia α = 0.05 → -log10(0.05) ≈ 1.301
linea_alpha = -np.log10(ALPHA)
ax.axhline(linea_alpha, color='black', lw=1.5, linestyle='--',
           label=f'α = {ALPHA} → -log₁₀(α) = {linea_alpha:.2f}')

for i, (bar, p, d) in enumerate(zip(bars, pvalores, decisiones)):
    ax.text(
        bar.get_x() + bar.get_width()/2,
        bar.get_height() + 0.15,
        f'p={p:.4f}\n{"✓" if "RECHAZAR" in d else "✗"} {d.split(" ")[0]}',
        ha='center', va='bottom', fontsize=8
    )

ax.set_xticks(range(4))
ax.set_xticklabels(nombres_tests, fontsize=9)
ax.set_ylabel('-log₁₀(p-value)', fontsize=10)
ax.set_title(
    'Resumen de Significancia Estadística — 4 Tests de Hipótesis\n'
    f'(Barras sobre la línea punteada → Rechazar H₀, α = {ALPHA})',
    fontsize=10
)
ax.legend(fontsize=9)
ax.grid(axis='y', alpha=0.3)

legend_patches = [
    mpatches.Patch(facecolor=COLOR_ACENTO, label='Rechazar H₀ (p < α)'),
    mpatches.Patch(facecolor='#AAAAAA', label='No rechazar H₀ (p ≥ α)'),
]
ax.legend(handles=legend_patches + ax.get_legend_handles_labels()[0], fontsize=9)

plt.tight_layout()
plt.savefig(PLOTS_DIR / 'test_resumen_decisiones.png', dpi=150, bbox_inches='tight')
plt.show()
print('Guardado: plots/test_resumen_decisiones.png')

## 6.2 Conclusión e Interpretación del OR Ajustado

In [ ]:
# ── Interpretación del OR ajustado ────────────────────────────────────────
print('ODDS RATIO AJUSTADO — FUE_TRASLADADO sobre MORTALIDAD_BINARIA')
print('='*60)
print(f'  OR            = {OR:.4f}')
print(f'  IC 95%        = [{OR_ci_lower:.4f}, {OR_ci_upper:.4f}]')
print(f'  β             = {beta_traslado:.6f}')
print(f'  SE(β)         = {se_traslado:.6f}')
print()

# Interpretación condicional
if OR > 1:
    incremento_pct = (OR - 1) * 100
    dir_efecto = 'AUMENTA'
    interpretacion_dir = 'mayor'
else:
    incremento_pct = (1 - OR) * 100
    dir_efecto = 'REDUCE'
    interpretacion_dir = 'menor'

print(f'Interpretación:')
print(f'  Un paciente de alta complejidad que FUE TRASLADADO tiene')
print(f'  {OR:.4f} veces las probabilidades de mortalidad alta respecto')
print(f'  a uno no trasladado, manteniendo constante PESO_GRD y región.')
print(f'  Esto representa un {dir_efecto} del {incremento_pct:.1f}% en las odds de')
print(f'  mortalidad alta para el grupo trasladado.')
print()

# ── Verificar coherencia entre tests ──────────────────────────────────────
tests_rechazan = sum('RECHAZAR' in d for d in decisiones)
print(f'Tests que rechazan H₀: {tests_rechazan}/4')

if tests_rechazan >= 3:
    evidencia = 'FUERTE'
elif tests_rechazan == 2:
    evidencia = 'MODERADA'
elif tests_rechazan == 1:
    evidencia = 'DÉBIL'
else:
    evidencia = 'INSUFICIENTE'

print(f'Nivel de evidencia: {evidencia}')

In [ ]:
# ── Párrafo de conclusión ─────────────────────────────────────────────────
conclusion = f"""
CONCLUSIÓN ESTADÍSTICA
{'='*70}

Los cuatro tests ejecutados aportan evidencia {evidencia} para la hipótesis central
del proyecto ADIE sobre desigualdades en derivaciones hospitalarias.

El Test 1 (Chi-cuadrado, χ²={chi2_stat:.2f}, p={p_chi2:.4f}) {'rechazó' if 'RECHAZAR' in decision_chi2 else 'no rechazó'} 
la hipótesis de independencia entre el traslado interhospitalario y la mortalidad alta 
en pacientes de alta complejidad (ALTA_SEVERIDAD=1), indicando que {'sí' if 'RECHAZAR' in decision_chi2 else 'no'} 
existe asociación estadística significativa entre ambas variables.

El Test 2 (Wald, z={z_score:.2f}, p={p_wald:.4f}) {'rechazó' if 'RECHAZAR' in decision_wald else 'no rechazó'} 
la H₀ sobre el coeficiente β de FUE_TRASLADADO en el modelo de regresión logística 
ajustado por PESO_GRD y región. El OR ajustado es {OR:.4f} (IC 95%: [{OR_ci_lower:.3f}, {OR_ci_upper:.3f}]), 
lo que {'sugiere que los pacientes trasladados tienen significativamente ' + interpretacion_dir + ' riesgo de mortalidad alta' if 'RECHAZAR' in decision_wald else 'no permite concluir un efecto significativo del traslado sobre la mortalidad'}.

El Test 3 (LRT={lrt_stat:.2f}, p={p_lrt:.4f}) {'confirmó' if 'RECHAZAR' in decision_lrt else 'no confirmó'} 
que incluir FUE_TRASLADADO mejora significativamente el ajuste del modelo respecto 
al modelo restringido sin esa variable (ΔAIC = {modelo_restringido.aic - modelo_completo.aic:.2f}), 
siendo este un test más robusto que el de Wald ante posibles problemas de convergencia.

El Test 4 (Mann-Whitney U={u_stat:.0f}, p={p_mw:.4f}) {'indicó' if 'RECHAZAR' in decision_mw else 'no evidenció'} 
que los pacientes trasladados presentan un PESO_GRD 
{'significativamente mayor' if 'RECHAZAR' in decision_mw else 'no significativamente diferente'} 
al de los no trasladados (medianas: {mediana_trasl:.3f} vs {mediana_no_trasl:.3f}), 
{'sugiriendo que los traslados corresponden a casos genuinamente más complejos, lo que es consistente con el mayor riesgo de mortalidad observado' if 'RECHAZAR' in decision_mw else 'lo que requiere mayor investigación sobre los determinantes del traslado'}.

DECISIÓN FINAL: Con {tests_rechazan}/4 tests rechazando H₀ (α=0.05) y una evidencia
{'se rechaza' if tests_rechazan >= 3 else 'no se puede rechazar con total certeza'} la hipótesis nula.
{'El traslado interhospitalario se asocia significativamente con un mayor riesgo de mortalidad en pacientes de alta complejidad del sistema GRD 2024.' if tests_rechazan >= 3 else 'La evidencia es insuficiente para concluir que el traslado interhospitalario aumenta significativamente la mortalidad en pacientes de alta complejidad.'}

NOTA METODOLÓGICA: La asociación observada es estadística, no causal. El efecto del
traslado puede estar parcialmente confundido por factores no observados (gravedad latente,
capacidad resolutiva del hospital emisor, características del trayecto). Para estimación
causal se requieren métodos como variables instrumentales o modelos de doble robustez.
"""
print(conclusion)

In [ ]:
# ── Exportar tabla resumen como CSV ───────────────────────────────────────
resumen_export = resumen_tests[['Test', 'Descripción', 'Estadístico',
                                'Valor_estadístico', 'p_value', 'Decisión', 'Efecto']].copy()

# Agregar metadatos del OR final
resumen_export.loc[len(resumen_export)] = {
    'Test': 'OR ajustado final',
    'Descripción': 'Odds Ratio de FUE_TRASLADADO ajustado por PESO_GRD y Región (Test 2)',
    'Estadístico': f'OR = {OR:.4f}',
    'Valor_estadístico': round(OR, 4),
    'p_value': round(p_wald, 6),
    'Decisión': f'IC 95%: [{OR_ci_lower:.4f}, {OR_ci_upper:.4f}]',
    'Efecto': f'β = {beta_traslado:.6f}, SE = {se_traslado:.6f}'
}

output_path = DATA_DIR / 'resumen_tests_hipotesis.csv'
resumen_export.to_csv(output_path, index=False, encoding='utf-8-sig')
print(f'✓ Tabla resumen exportada: {output_path}')
print()
print(resumen_export.to_string(index=False))